# Stage 3 - Modeling Next-Night Sleep Outcomes


## Framing

This notebook keeps the original `day D -> next night` contract, but stops assuming that one binary cutoff (`recovery < 70`) is the only sensible modeling target.

What this notebook now does:
- screens several recovery thresholds (`70`, `75`, `80`, and the empirical median) for the binary classification task
- keeps the main classification story tied to next-night recovery risk
- compares several continuous next-night targets with different distributions: `sleepRecoveryScore`, `sleepOverallScore`, `sleepQualityScore`, and `avgSleepStress`

The goal is not to force one device metric to work. The goal is to see which target/task pair produces the clearest and most defensible Stage 3 signal.


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

repo_root = Path.cwd().resolve()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from garmin_analytics.modeling.prepare import (
    DEFAULT_LOW_RECOVERY_THRESHOLD,
    load_recovery_modeling_frame,
    resolve_recovery_feature_columns,
)
from garmin_analytics.modeling.recovery import (
    build_confusion_table,
    build_feature_importance_table,
    build_permutation_importance_table,
    evaluate_recovery_classification_models_tuned,
    evaluate_recovery_regression_models,
    forward_select_classification_features,
)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)


In [2]:
daily_path = repo_root / 'data' / 'processed' / 'daily_sanitized.parquet'
quality_path = repo_root / 'data' / 'processed' / 'daily_quality.parquet'
DEFAULT_RECOVERY_THRESHOLD = DEFAULT_LOW_RECOVERY_THRESHOLD
RECOVERY_SCREEN_THRESHOLDS = [70.0, 75.0, 80.0]
PRIMARY_RECOVERY_THRESHOLD = 75.0

print('repo_root:', repo_root)
print('daily_path exists:', daily_path.exists(), daily_path)
print('quality_path exists:', quality_path.exists(), quality_path)
print('default recovery threshold:', DEFAULT_RECOVERY_THRESHOLD)
print('recovery thresholds to screen:', RECOVERY_SCREEN_THRESHOLDS)
print('primary recovery threshold for deep-dive:', PRIMARY_RECOVERY_THRESHOLD)


repo_root: /Users/abatrakov/Documents/FUN/wearable-analytics
daily_path exists: True /Users/abatrakov/Documents/FUN/wearable-analytics/data/processed/daily_sanitized.parquet
quality_path exists: True /Users/abatrakov/Documents/FUN/wearable-analytics/data/processed/daily_quality.parquet
default recovery threshold: 70.0
recovery thresholds to screen: [70.0, 75.0, 80.0]
primary recovery threshold for deep-dive: 75.0


In [3]:
model_df = load_recovery_modeling_frame(
    daily_path,
    quality_path,
    current_day_quality='strict',
    low_recovery_threshold=DEFAULT_RECOVERY_THRESHOLD,
    include_schedule_targets=True,
)

recovery_score = pd.to_numeric(model_df['target_sleepRecoveryScore_next_night'], errors='coerce')
recovery_median = float(recovery_score.dropna().median())
recovery_thresholds = []
seen_thresholds = set()
for threshold in [*RECOVERY_SCREEN_THRESHOLDS, recovery_median]:
    threshold = float(threshold)
    if threshold not in seen_thresholds:
        recovery_thresholds.append(threshold)
        seen_thresholds.add(threshold)

summary = pd.DataFrame(
    [
        {
            'rows_total': len(model_df),
            'rows_with_recovery_target': int(model_df['target_sleepRecoveryScore_next_night'].notna().sum()),
            'rows_with_overall_target': int(model_df['target_sleepOverallScore_next_night'].notna().sum()),
            'rows_with_quality_target': int(model_df['target_sleepQualityScore_next_night'].notna().sum()),
            'rows_with_avg_sleep_stress_target': int(model_df['target_avgSleepStress_next_night'].notna().sum()),
            'recovery_median': recovery_median,
        }
    ]
)

recovery_threshold_summary = pd.DataFrame(
    [
        {
            'threshold': threshold,
            'positive_rate_lt_threshold': float((recovery_score < threshold).mean()),
        }
        for threshold in recovery_thresholds
    ]
)

sample_cols = [
    'calendarDate',
    'target_sleepRecoveryScore_next_night',
    'target_sleepOverallScore_next_night',
    'target_sleepQualityScore_next_night',
    'target_avgSleepStress_next_night',
    'awakeAverageStressLevel',
    'bodyBatteryLowest',
    'active_hours',
    'nextsleep_sleep_start_hour_local_wrapped',
    'nextsleep_sleep_opportunity_hours',
]

available_sample_cols = [col for col in sample_cols if col in model_df.columns]

display(summary)
display(recovery_threshold_summary)
display(model_df[available_sample_cols].head(12))


,rows_total,rows_with_recovery_target,rows_with_overall_target,rows_with_quality_target,rows_with_avg_sleep_stress_target,recovery_median
0,525,464,464,464,465,79.0


,threshold,positive_rate_lt_threshold
0,70.0,0.191810
1,75.0,0.338362
2,80.0,0.512931
3,79.0,0.484914


,calendarDate,target_sleepRecoveryScore_next_night,target_sleepOverallScore_next_night,target_sleepQualityScore_next_night,target_avgSleepStress_next_night,awakeAverageStressLevel,bodyBatteryLowest,active_hours,nextsleep_sleep_start_hour_local_wrapped,nextsleep_sleep_opportunity_hours
0,2023-05-26,100,98,97,5.48,50,36,0.356944,2.800000,7.9
1,2023-05-27,100,82,88,8.59,41,22,1.7925,3.383333,7.549722
2,2023-05-28,100,90,89,8.02,58,8,0.366944,2.583333,8.816667
3,2023-05-29,100,85,93,6.59,36,18,1.527222,2.950000,7.133333
4,2023-05-30,87,69,75,13.16,38,24,0.936667,1.250000,11.532778
5,2023-05-31,100,87,85,7.44,46,22,0.638889,1.883333,8.066667
6,2023-06-01,100,67,93,8.79,51,20,1.285278,2.183333,5.25
7,2023-06-02,100,73,93,8.94,42,12,1.223611,6.933333,5.766667
8,2023-06-03,100,93,93,7.58,40,6,2.181667,4.350000,9.733333
9,2023-06-04,100,96,95,3.55,70,5,0.668056,1.883333,8.433333


## Feature sets


In [4]:
awake_only_cols = resolve_recovery_feature_columns(
    model_df,
    tiers=('tier1',),
    include_schedule=False,
)
awake_plus_compact_stress_cols = resolve_recovery_feature_columns(
    model_df,
    tiers=('tier1', 'tier2_compact'),
    include_schedule=False,
)
awake_plus_stress_cols = resolve_recovery_feature_columns(
    model_df,
    tiers=('tier1', 'tier2'),
    include_schedule=False,
)
awake_plus_compact_stress_plus_schedule_cols = resolve_recovery_feature_columns(
    model_df,
    tiers=('tier1', 'tier2_compact'),
    include_schedule=True,
)

feature_sets = {
    'awake_only': awake_only_cols,
    'awake_plus_compact_stress': awake_plus_compact_stress_cols,
    'awake_plus_stress': awake_plus_stress_cols,
    'awake_plus_compact_stress_plus_schedule': awake_plus_compact_stress_plus_schedule_cols,
}

screening_feature_sets = {
    name: cols
    for name, cols in feature_sets.items()
    if name != 'awake_plus_compact_stress_plus_schedule'
}

display(
    pd.DataFrame(
        [
            {
                'feature_set': name,
                'feature_count': len(cols),
                'sample_features': ', '.join(cols[:10]),
            }
            for name, cols in feature_sets.items()
        ]
    )
)


,feature_set,feature_count,sample_features
0,awake_only,21,"totalSteps, totalDistanceMeters, activeKilocal..."
1,awake_plus_compact_stress,26,"totalSteps, totalDistanceMeters, activeKilocal..."
2,awake_plus_stress,45,"totalSteps, totalDistanceMeters, activeKilocal..."
3,awake_plus_compact_stress_plus_schedule,28,"totalSteps, totalDistanceMeters, activeKilocal..."


## Recovery threshold screening


In [5]:
def add_threshold_target(frame: pd.DataFrame, *, source_col: str, threshold: float, target_col: str) -> pd.DataFrame:
    out = frame.copy()
    score = pd.to_numeric(out[source_col], errors='coerce')
    out[target_col] = score.lt(threshold).where(score.notna(), pd.NA).astype('Int64')
    return out

recovery_threshold_results = []

for threshold in recovery_thresholds:
    target_col = f'target_low_recovery_lt_{str(threshold).replace(".", "_")}'
    threshold_df = add_threshold_target(
        model_df,
        source_col='target_sleepRecoveryScore_next_night',
        threshold=threshold,
        target_col=target_col,
    )
    for feature_set_name, cols in screening_feature_sets.items():
        out, _ = evaluate_recovery_classification_models_tuned(
            threshold_df,
            feature_cols=cols,
            target_col=target_col,
            threshold_metric='balanced_accuracy',
        )
        out['feature_set'] = feature_set_name
        out['recovery_threshold'] = threshold
        recovery_threshold_results.append(out)

recovery_threshold_results = pd.concat(recovery_threshold_results, ignore_index=True)
recovery_threshold_test = recovery_threshold_results[recovery_threshold_results['split'] == 'test'].copy()
recovery_threshold_test = recovery_threshold_test.sort_values(
    ['recovery_threshold', 'balanced_accuracy', 'roc_auc', 'pr_auc'],
    ascending=[True, False, False, False],
)

best_by_threshold = recovery_threshold_test.drop_duplicates(['recovery_threshold'], keep='first')

display(recovery_threshold_test)
display(best_by_threshold)


,model,split,n_rows,feature_count,threshold,threshold_metric,valid_metric_at_threshold,accuracy,balanced_accuracy,f1,positive_rate_true,positive_rate_pred,roc_auc,pr_auc,feature_set,recovery_threshold
35,hist_gradient_boosting,test,93,26,0.15,balanced_accuracy,0.647205,0.483871,0.637432,0.333333,0.150538,0.623656,0.674503,0.313507,awake_plus_compact_stress,70.0
50,random_forest,test,93,45,0.30,balanced_accuracy,0.690373,0.516129,0.627034,0.328358,0.150538,0.569892,0.650090,0.282914,awake_plus_stress,70.0
5,logistic_regression,test,93,21,0.70,balanced_accuracy,0.603106,0.731183,0.606691,0.324324,0.150538,0.247312,0.656420,0.296100,awake_only,70.0
32,random_forest,test,93,26,0.25,balanced_accuracy,0.677019,0.430108,0.605787,0.311688,0.150538,0.677419,0.622966,0.318214,awake_plus_compact_stress,70.0
23,logistic_regression,test,93,26,0.65,balanced_accuracy,0.610559,0.569892,0.599910,0.310345,0.150538,0.473118,0.611212,0.192076,awake_plus_compact_stress,70.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110,dummy_most_frequent,test,93,21,0.50,balanced_accuracy,0.500000,0.451613,0.500000,0.622222,0.451613,1.000000,0.500000,0.451613,awake_only,80.0
128,dummy_most_frequent,test,93,26,0.50,balanced_accuracy,0.500000,0.451613,0.500000,0.622222,0.451613,1.000000,0.500000,0.451613,awake_plus_compact_stress,80.0
146,dummy_most_frequent,test,93,45,0.50,balanced_accuracy,0.500000,0.451613,0.500000,0.622222,0.451613,1.000000,0.500000,0.451613,awake_plus_stress,80.0
116,logistic_regression_l1,test,93,21,0.50,balanced_accuracy,0.565631,0.462366,0.482493,0.537037,0.451613,0.709677,0.563025,0.544478,awake_only,80.0


,model,split,n_rows,feature_count,threshold,threshold_metric,valid_metric_at_threshold,accuracy,balanced_accuracy,f1,positive_rate_true,positive_rate_pred,roc_auc,pr_auc,feature_set,recovery_threshold
35,hist_gradient_boosting,test,93,26,0.15,balanced_accuracy,0.647205,0.483871,0.637432,0.333333,0.150538,0.623656,0.674503,0.313507,awake_plus_compact_stress,70.0
83,logistic_regression_elasticnet,test,93,26,0.65,balanced_accuracy,0.626869,0.548387,0.610115,0.432432,0.236559,0.559140,0.597311,0.322725,awake_plus_compact_stress,75.0
188,logistic_regression_l1,test,93,26,0.60,balanced_accuracy,0.618738,0.612903,0.619748,0.617021,0.451613,0.559140,0.621382,0.625086,awake_plus_compact_stress,79.0
134,logistic_regression_l1,test,93,26,0.55,balanced_accuracy,0.666512,0.580645,0.590336,0.597938,0.451613,0.591398,0.617180,0.603456,awake_plus_compact_stress,80.0


## Selected recovery threshold deep-dive


In [6]:
selected_target_col = f'target_low_recovery_lt_{str(PRIMARY_RECOVERY_THRESHOLD).replace(".", "_")}'
model_df_selected = add_threshold_target(
    model_df,
    source_col='target_sleepRecoveryScore_next_night',
    threshold=PRIMARY_RECOVERY_THRESHOLD,
    target_col=selected_target_col,
)

classification_tuned_results = []
classification_fitted_by_feature_set = {}

for feature_set_name, cols in feature_sets.items():
    out, fitted = evaluate_recovery_classification_models_tuned(
        model_df_selected,
        feature_cols=cols,
        target_col=selected_target_col,
        threshold_metric='balanced_accuracy',
    )
    out['feature_set'] = feature_set_name
    out['recovery_threshold'] = PRIMARY_RECOVERY_THRESHOLD
    classification_tuned_results.append(out)
    classification_fitted_by_feature_set[feature_set_name] = fitted

classification_tuned_results = pd.concat(classification_tuned_results, ignore_index=True)
display(
    classification_tuned_results[classification_tuned_results['split'] == 'test']
    .sort_values(['balanced_accuracy', 'roc_auc', 'pr_auc'], ascending=[False, False, False])
)


,model,split,n_rows,feature_count,threshold,threshold_metric,valid_metric_at_threshold,accuracy,balanced_accuracy,f1,positive_rate_true,positive_rate_pred,roc_auc,pr_auc,feature_set,recovery_threshold
71,hist_gradient_boosting,test,93,28,0.25,balanced_accuracy,0.587488,0.537634,0.634443,0.455696,0.236559,0.612903,0.597951,0.281522,awake_plus_compact_stress_plus_schedule,75.0
29,logistic_regression_elasticnet,test,93,26,0.65,balanced_accuracy,0.626869,0.548387,0.610115,0.432432,0.236559,0.559140,0.597311,0.322725,awake_plus_compact_stress,75.0
68,random_forest,test,93,28,0.40,balanced_accuracy,0.558574,0.505376,0.597631,0.425000,0.236559,0.623656,0.579385,0.273028,awake_plus_compact_stress_plus_schedule,75.0
23,logistic_regression,test,93,26,0.65,balanced_accuracy,0.601446,0.526882,0.596031,0.421053,0.236559,0.580645,0.590269,0.317390,awake_plus_compact_stress,75.0
65,logistic_regression_elasticnet,test,93,28,0.65,balanced_accuracy,0.603689,0.548387,0.594430,0.416667,0.236559,0.537634,0.616517,0.337096,awake_plus_compact_stress_plus_schedule,75.0
32,random_forest,test,93,26,0.35,balanced_accuracy,0.581256,0.397849,0.589949,0.428571,0.236559,0.817204,0.556338,0.263256,awake_plus_compact_stress,75.0
35,hist_gradient_boosting,test,93,26,0.20,balanced_accuracy,0.559821,0.440860,0.586748,0.422222,0.236559,0.731183,0.573624,0.265927,awake_plus_compact_stress,75.0
59,logistic_regression,test,93,28,0.60,balanced_accuracy,0.588485,0.430108,0.579706,0.417582,0.236559,0.741935,0.607554,0.309218,awake_plus_compact_stress_plus_schedule,75.0
62,logistic_regression_l1,test,93,28,0.65,balanced_accuracy,0.639831,0.569892,0.577145,0.393939,0.236559,0.473118,0.619718,0.324116,awake_plus_compact_stress_plus_schedule,75.0
26,logistic_regression_l1,test,93,26,0.65,balanced_accuracy,0.625125,0.569892,0.577145,0.393939,0.236559,0.473118,0.613956,0.321173,awake_plus_compact_stress,75.0


## Automated feature selection


In [7]:
selection_history_logistic, selected_features_logistic = forward_select_classification_features(
    model_df_selected,
    candidate_feature_cols=feature_sets['awake_plus_compact_stress'],
    target_col=selected_target_col,
    model_name='logistic_regression',
    selection_metric='balanced_accuracy',
    threshold_metric='balanced_accuracy',
    max_features=8,
    min_improvement=0.01,
)

selection_history_l1, selected_features_l1 = forward_select_classification_features(
    model_df_selected,
    candidate_feature_cols=feature_sets['awake_plus_compact_stress'],
    target_col=selected_target_col,
    model_name='logistic_regression_l1',
    selection_metric='balanced_accuracy',
    threshold_metric='balanced_accuracy',
    max_features=8,
    min_improvement=0.01,
)

selected_feature_sets = {
    'forward_selected_logistic': selected_features_logistic,
    'forward_selected_logistic_l1': selected_features_l1,
}

selected_feature_set_results = []
selected_feature_set_fitted = {}
for feature_set_name, cols in selected_feature_sets.items():
    out, fitted = evaluate_recovery_classification_models_tuned(
        model_df_selected,
        feature_cols=cols,
        target_col=selected_target_col,
        threshold_metric='balanced_accuracy',
        model_names=['logistic_regression', 'logistic_regression_l1', 'hist_gradient_boosting', 'random_forest'],
    )
    out['feature_set'] = feature_set_name
    selected_feature_set_results.append(out)
    selected_feature_set_fitted[feature_set_name] = fitted

selected_feature_set_results = pd.concat(selected_feature_set_results, ignore_index=True)

display(selection_history_logistic)
display(selection_history_l1)
display(
    pd.DataFrame(
        [
            {'feature_set': name, 'feature_count': len(cols), 'features': ', '.join(cols)}
            for name, cols in selected_feature_sets.items()
        ]
    )
)
display(
    selected_feature_set_results[selected_feature_set_results['split'] == 'test']
    .sort_values(['balanced_accuracy', 'roc_auc', 'pr_auc'], ascending=[False, False, False])
)


,step,model,selection_metric,added_feature,n_selected_features,selected_features,candidate_score,improvement_from_prev,threshold,valid_balanced_accuracy,valid_f1,valid_roc_auc,valid_pr_auc,valid_positive_rate_pred
0,1,logistic_regression,balanced_accuracy,awakeAverageStressLevel,1,awakeAverageStressLevel,0.664756,NaN,0.55,0.664756,0.583333,0.720837,0.577535,0.408602
1,2,logistic_regression,balanced_accuracy,maxHeartRate,2,"awakeAverageStressLevel, maxHeartRate",0.706630,0.041874,0.55,0.706630,0.641026,0.748504,0.652698,0.473118


,step,model,selection_metric,added_feature,n_selected_features,selected_features,candidate_score,improvement_from_prev,threshold,valid_balanced_accuracy,valid_f1,valid_roc_auc,valid_pr_auc,valid_positive_rate_pred
0,1,logistic_regression_l1,balanced_accuracy,awakeAverageStressLevel,1,awakeAverageStressLevel,0.664756,NaN,0.55,0.664756,0.583333,0.720837,0.577535,0.408602
1,2,logistic_regression_l1,balanced_accuracy,maxHeartRate,2,"awakeAverageStressLevel, maxHeartRate",0.694167,0.029412,0.55,0.694167,0.621622,0.746012,0.646922,0.430108
2,3,logistic_regression_l1,balanced_accuracy,bodyBatteryLowest,3,"awakeAverageStressLevel, maxHeartRate, bodyBat...",0.708873,0.014706,0.55,0.708873,0.640000,0.740279,0.645131,0.440860


,feature_set,feature_count,features
0,forward_selected_logistic,2,"awakeAverageStressLevel, maxHeartRate"
1,forward_selected_logistic_l1,3,"awakeAverageStressLevel, maxHeartRate, bodyBat..."


,model,split,n_rows,feature_count,threshold,threshold_metric,valid_metric_at_threshold,accuracy,balanced_accuracy,f1,positive_rate_true,positive_rate_pred,roc_auc,pr_auc,feature_set
14,logistic_regression,test,93,3,0.55,balanced_accuracy,0.689681,0.505376,0.644686,0.465116,0.236559,0.688172,0.642766,0.349396,forward_selected_logistic_l1
17,logistic_regression_l1,test,93,3,0.55,balanced_accuracy,0.708873,0.526882,0.643086,0.463415,0.236559,0.645161,0.649808,0.345458,forward_selected_logistic_l1
20,hist_gradient_boosting,test,93,3,0.40,balanced_accuracy,0.643320,0.494624,0.637644,0.459770,0.236559,0.698925,0.688540,0.394308,forward_selected_logistic_l1
8,hist_gradient_boosting,test,93,2,0.40,balanced_accuracy,0.613410,0.494624,0.637644,0.459770,0.236559,0.698925,0.675416,0.400033,forward_selected_logistic
5,logistic_regression_l1,test,93,2,0.55,balanced_accuracy,0.694167,0.526882,0.611716,0.435897,0.236559,0.602151,0.642766,0.334852,forward_selected_logistic
2,logistic_regression,test,93,2,0.55,balanced_accuracy,0.706630,0.483871,0.599232,0.428571,0.236559,0.666667,0.632522,0.337624,forward_selected_logistic
11,random_forest,test,93,2,0.40,balanced_accuracy,0.585743,0.397849,0.574264,0.416667,0.236559,0.795699,0.547375,0.346946,forward_selected_logistic
23,random_forest,test,93,3,0.40,balanced_accuracy,0.598205,0.365591,0.568822,0.415842,0.236559,0.849462,0.596671,0.377026,forward_selected_logistic_l1


## Main candidate: sparse logistic plus nonlinear benchmark


In [8]:
main_feature_set = 'forward_selected_logistic_l1'
main_model_name = 'logistic_regression_l1'

main_test_row = (
    selected_feature_set_results[
        (selected_feature_set_results['feature_set'] == main_feature_set)
        & (selected_feature_set_results['model'] == main_model_name)
        & (selected_feature_set_results['split'] == 'test')
    ]
    .iloc[0]
)
main_threshold = float(main_test_row['threshold'])
main_feature_cols = selected_feature_sets[main_feature_set]
main_pipeline = selected_feature_set_fitted[main_feature_set][main_model_name]

frame_for_split = model_df_selected[['calendarDate', selected_target_col, *main_feature_cols]].copy()
frame_for_split = frame_for_split.dropna(subset=[selected_target_col])
frame_for_split[selected_target_col] = pd.to_numeric(frame_for_split[selected_target_col], errors='coerce').astype(int)
frame_for_split['calendarDate'] = pd.to_datetime(frame_for_split['calendarDate'], errors='coerce')
frame_for_split = frame_for_split.sort_values('calendarDate').reset_index(drop=True)

n_rows = len(frame_for_split)
train_end = max(1, int(n_rows * 0.6))
valid_end = max(train_end + 1, int(n_rows * 0.8))
valid_end = min(valid_end, n_rows - 1)
test_df = frame_for_split.iloc[valid_end:].copy()

test_proba = main_pipeline.predict_proba(test_df[main_feature_cols])[:, 1]
confusion_tbl = build_confusion_table(
    test_df[selected_target_col],
    test_proba,
    threshold=main_threshold,
)

display(main_test_row.to_frame().T)
display(confusion_tbl)

logistic_pipeline = selected_feature_set_fitted['forward_selected_logistic_l1']['logistic_regression_l1']
logistic_importance_tbl = build_feature_importance_table(logistic_pipeline, top_n=12)
print('Feature importance for forward_selected_logistic_l1 / logistic_regression_l1')
display(logistic_importance_tbl)

hgb_feature_set = 'forward_selected_logistic'
hgb_model_name = 'hist_gradient_boosting'
hgb_feature_cols = selected_feature_sets[hgb_feature_set]
hgb_pipeline = selected_feature_set_fitted[hgb_feature_set][hgb_model_name]

hgb_frame_for_split = model_df_selected[['calendarDate', selected_target_col, *hgb_feature_cols]].copy()
hgb_frame_for_split = hgb_frame_for_split.dropna(subset=[selected_target_col])
hgb_frame_for_split[selected_target_col] = pd.to_numeric(hgb_frame_for_split[selected_target_col], errors='coerce').astype(int)
hgb_frame_for_split['calendarDate'] = pd.to_datetime(hgb_frame_for_split['calendarDate'], errors='coerce')
hgb_frame_for_split = hgb_frame_for_split.sort_values('calendarDate').reset_index(drop=True)

hgb_n_rows = len(hgb_frame_for_split)
hgb_train_end = max(1, int(hgb_n_rows * 0.6))
hgb_valid_end = max(hgb_train_end + 1, int(hgb_n_rows * 0.8))
hgb_valid_end = min(hgb_valid_end, hgb_n_rows - 1)
hgb_test_df = hgb_frame_for_split.iloc[hgb_valid_end:].copy()

hgb_importance_tbl = build_permutation_importance_table(
    hgb_pipeline,
    hgb_test_df[hgb_feature_cols],
    hgb_test_df[selected_target_col],
    top_n=12,
    scoring='balanced_accuracy',
    n_repeats=20,
)
print('Permutation importance for forward_selected_logistic / hist_gradient_boosting')
display(hgb_importance_tbl)


,model,split,n_rows,feature_count,threshold,threshold_metric,valid_metric_at_threshold,accuracy,balanced_accuracy,f1,positive_rate_true,positive_rate_pred,roc_auc,pr_auc,feature_set
17,logistic_regression_l1,test,93,3,0.55,balanced_accuracy,0.708873,0.526882,0.643086,0.463415,0.236559,0.645161,0.649808,0.345458,forward_selected_logistic_l1


,threshold,tn,fp,fn,tp,precision,recall,specificity
0,0.55,30,41,3,19,0.316667,0.863636,0.422535


Feature importance for forward_selected_logistic_l1 / logistic_regression_l1


,feature,coefficient,importance_abs,direction
0,awakeAverageStressLevel,0.375556,0.375556,positive
1,maxHeartRate,-0.224770,0.224770,negative
2,bodyBatteryLowest,-0.044182,0.044182,negative


Permutation importance for forward_selected_logistic / hist_gradient_boosting


,feature,importance_mean,importance_std
0,awakeAverageStressLevel,0.110339,0.040552
1,maxHeartRate,0.041949,0.048925


## Alternative regression targets


In [9]:
regression_feature_sets = {
    'awake_only': feature_sets['awake_only'],
    'awake_plus_compact_stress': feature_sets['awake_plus_compact_stress'],
}

regression_target_specs = [
    {
        'target_col': 'target_sleepRecoveryScore_next_night',
        'target_label': 'sleepRecoveryScore',
        'target_bounds': (0.0, 100.0),
    },
    {
        'target_col': 'target_sleepOverallScore_next_night',
        'target_label': 'sleepOverallScore',
        'target_bounds': (0.0, 100.0),
    },
    {
        'target_col': 'target_sleepQualityScore_next_night',
        'target_label': 'sleepQualityScore',
        'target_bounds': (0.0, 100.0),
    },
    {
        'target_col': 'target_avgSleepStress_next_night',
        'target_label': 'avgSleepStress',
        'target_bounds': None,
    },
]

regression_results = []
for spec in regression_target_specs:
    for feature_set_name, cols in regression_feature_sets.items():
        out = evaluate_recovery_regression_models(
            model_df,
            feature_cols=cols,
            target_col=spec['target_col'],
            target_bounds=spec['target_bounds'],
        )
        out['feature_set'] = feature_set_name
        out['target_label'] = spec['target_label']
        regression_results.append(out)

regression_results = pd.concat(regression_results, ignore_index=True)
regression_test = regression_results[regression_results['split'] == 'test'].copy()
regression_test = regression_test.sort_values(['target_label', 'r2', 'mae'], ascending=[True, False, True])

best_regression_by_target = regression_test.drop_duplicates(['target_label'], keep='first')

display(regression_test)
display(best_regression_by_target)


,model,split,n_rows,feature_count,target_bounds,mae,rmse,r2,pred_min,pred_max,feature_set,target_label
110,dummy_median,test,93,21,none,4.207742,5.375593,-0.026632,15.770000,15.770000,awake_only,avgSleepStress
128,dummy_median,test,93,26,none,4.207742,5.375593,-0.026632,15.770000,15.770000,awake_plus_compact_stress,avgSleepStress
122,random_forest,test,93,21,none,4.238256,5.428116,-0.046792,11.819779,21.352049,awake_only,avgSleepStress
125,hist_gradient_boosting,test,93,21,none,4.460384,5.700195,-0.154360,10.925424,25.633990,awake_only,avgSleepStress
113,ridge,test,93,21,none,4.555443,5.737189,-0.169393,11.302462,22.592036,awake_only,avgSleepStress
119,elastic_net,test,93,21,none,4.582761,5.766177,-0.181239,11.217626,22.669455,awake_only,avgSleepStress
116,lasso,test,93,21,none,4.600047,5.784412,-0.188723,11.015020,22.767491,awake_only,avgSleepStress
143,hist_gradient_boosting,test,93,26,none,5.403032,6.443782,-0.475176,10.508307,26.804750,awake_plus_compact_stress,avgSleepStress
140,random_forest,test,93,26,none,5.348479,6.620777,-0.557328,12.831230,27.644859,awake_plus_compact_stress,avgSleepStress
137,elastic_net,test,93,26,none,5.652466,6.780774,-0.633505,14.171123,27.252263,awake_plus_compact_stress,avgSleepStress


,model,split,n_rows,feature_count,target_bounds,mae,rmse,r2,pred_min,pred_max,feature_set,target_label
110,dummy_median,test,93,21,none,4.207742,5.375593,-0.026632,15.77,15.77,awake_only,avgSleepStress
38,dummy_median,test,93,21,"(0.0, 100.0)",10.021505,11.772028,-0.260898,81.00,81.00,awake_only,sleepOverallScore
74,dummy_median,test,93,21,"(0.0, 100.0)",8.021505,9.726364,-0.309951,83.00,83.00,awake_only,sleepQualityScore
2,dummy_median,test,93,21,"(0.0, 100.0)",12.241935,17.315929,-0.047377,78.50,78.50,awake_only,sleepRecoveryScore


## Interpretation checklist

After rerunning the notebook, use the outputs above to answer five questions:

- Which recovery threshold gives the best balance between practical meaning and class balance: `70`, `75`, `80`, or the empirical median?
- Do the new model families (`ElasticNet logistic`, `HistGradientBoosting`) materially improve the test split over the original logistic/random-forest pair?
- Does automated forward selection produce a smaller feature set without collapsing test performance?
- Is the best result better framed as an interpretable sparse logistic model, or as a slightly stronger nonlinear benchmark?
- Is any continuous next-night target now clearly predictable, or does classification remain the only robust Stage 3 result?

Current expectation before seeing fresh outputs:
- classification should remain the strongest story
- `recovery < 75` should still be the best practical cutoff
- `ElasticNet logistic` and `HistGradientBoosting` may outperform the original baseline models modestly
- forward selection should make the model narrative cleaner even if it does not create a dramatic jump in performance
- continuous targets may still remain weaker than the binary risk task
